# 07b - Optuna sobre LightGBM + BERT/SVD

Este notebook continua el apartado de modelado despues de seleccionar `lightgbm_final_bert` como arquitectura ganadora en el notebook 07.

Objetivo: comprobar si una busqueda adicional de hiperparametros mejora el modelo ganador usando solo `train`, validacion cruzada agrupada por paciente y predicciones OOF. El test temporal no se carga ni se utiliza en este notebook.

## 1. Protocolo

- Modelo estudiado: LightGBM con variables tabulares finales + componentes BERT/SVD.
- Datos: `X_train_final`, `bert_embeddings_train`, `y_train`, `groups_train`, `sample_weights_train`.
- Validacion: `StratifiedGroupKFold` agrupado por paciente.
- Objetivo Optuna: maximizar Macro F1 OOF medio.
- Metrica principal del modelo congelado actual: Macro F1 OOF = `0.567940`.
- Regla metodologica: no se usa test temporal para seleccionar hiperparametros, features, thresholds ni politica.

In [1]:
# ruff: noqa: E402, I001
import json
import sys
import time
import warnings
from pathlib import Path
from typing import Any

import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("No se pudo localizar la raiz del proyecto.")
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from triaje_ia.config import DATA_PROCESSED, MODELS_DIR, REPORTS_DIR

RANDOM_STATE = 42
CLASSES = np.array([1, 2, 3, 4, 5])
BASELINE_MACRO_F1 = 0.567940
N_SPLITS = 5
N_TRIALS = 60
TIMEOUT_SECONDS = None

OUT_DIR = REPORTS_DIR / "hyperparameter_tuning"
OUT_DIR.mkdir(parents=True, exist_ok=True)

STUDY_NAME = "lgbm_bert_optuna_macro_f1_oof"
STORAGE_URL = f"sqlite:///{(OUT_DIR / 'lgbm_bert_optuna_study.db').as_posix()}"

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Salida: {OUT_DIR}")

Proyecto: c:\Users\CARLOS\triaje-ia-tfg
Salida: C:\Users\CARLOS\triaje-ia-tfg\reports\hyperparameter_tuning


c:\Users\CARLOS\triaje-ia-tfg\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Carga de train y comprobaciones

Se reconstruye la misma matriz del modelo final, pero solo para train. La lista de columnas se carga desde `models/feature_list.json` para respetar el contrato congelado del modelo actual.

In [2]:
required = [
    DATA_PROCESSED / "X_train_final.parquet",
    DATA_PROCESSED / "bert_embeddings_train.parquet",
    DATA_PROCESSED / "y_train.parquet",
    DATA_PROCESSED / "groups_train.npy",
    DATA_PROCESSED / "sample_weights_train.npy",
    MODELS_DIR / "feature_list.json",
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Faltan artefactos: " + ", ".join(map(str, missing)))

feature_payload = json.loads((MODELS_DIR / "feature_list.json").read_text(encoding="utf-8"))
feature_list = feature_payload["features"]

X_tab = pd.read_parquet(DATA_PROCESSED / "X_train_final.parquet").reset_index(drop=True)
X_bert = pd.read_parquet(DATA_PROCESSED / "bert_embeddings_train.parquet").reset_index(drop=True)
X_train = pd.concat([X_tab, X_bert], axis=1)[feature_list]
y_train = pd.read_parquet(DATA_PROCESSED / "y_train.parquet").squeeze().astype(int).reset_index(drop=True)
groups_train = np.load(DATA_PROCESSED / "groups_train.npy")
sample_weights_train = np.load(DATA_PROCESSED / "sample_weights_train.npy")

assert X_train.shape[0] == y_train.shape[0] == len(groups_train) == len(sample_weights_train)
assert X_train.shape[1] == len(feature_list) == 79
assert y_train.isin(CLASSES).all()
assert (sample_weights_train > 0).all() and not np.isnan(sample_weights_train).any()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("Pacientes/grupos:", pd.Series(groups_train).nunique())
print("Distribucion train:")
print(y_train.value_counts(normalize=True).sort_index().round(4))

X_train: (334480, 79)
y_train: (334480,)
Pacientes/grupos: 168596
Distribucion train:
acuity
1    0.0579
2    0.3326
3    0.5372
4    0.0695
5    0.0028
Name: proportion, dtype: float64


## 3. Analisis del desbalance en train

Aunque Acuity 1 tiene un porcentaje visible en train, el problema sigue estando desbalanceado: la clase mayoritaria es Acuity 3 y la clase Acuity 5 es extremadamente minoritaria. Este analisis justifica el uso de Macro F1, metricas por clase y `sample_weights_train`.

La tabla siguiente combina frecuencia real, ratio frente a la clase mayoritaria y peso efectivo tras aplicar `sample_weights_train`.

In [3]:
counts = y_train.value_counts().sort_index()
pct = counts / len(y_train) * 100
majority = counts.max()

imbalance_rows = []
for acuity in counts.index:
    mask = y_train.to_numpy() == acuity
    imbalance_rows.append(
        {
            "acuity": int(acuity),
            "n": int(counts.loc[acuity]),
            "pct_train": float(pct.loc[acuity]),
            "ratio_mayoritaria_vs_clase": float(majority / counts.loc[acuity]),
            "sample_weight_medio": float(sample_weights_train[mask].mean()),
            "peso_total_clase": float(sample_weights_train[mask].sum()),
            "pct_efectivo_tras_pesos": float(sample_weights_train[mask].sum() / sample_weights_train.sum() * 100),
        }
    )

imbalance_df = pd.DataFrame(imbalance_rows)
display(imbalance_df)

print(f"Ratio mayoria/minoria: {counts.max():,} / {counts.min():,} = {counts.max() / counts.min():.1f}:1")
print(f"Acuity 1-2 en train: {pct.loc[[1, 2]].sum():.2f}%")
print(f"Acuity 4-5 en train: {pct.loc[[4, 5]].sum():.2f}%")
print(f"Acuity 5 por fold aprox. en CV de 5 folds: {counts.loc[5] / 5:.1f} casos")

,acuity,n,pct_train,ratio_mayoritaria_vs_clase,sample_weight_medio,peso_total_clase,pct_efectivo_tras_pesos
0,1,19371,5.791378,9.275463,2.226040,43120.627355,12.891840
1,2,111262,33.264171,1.614882,0.928829,103343.337082,30.896716
2,3,179675,53.717711,1.000000,0.730912,131326.653291,39.262932
3,4,23240,6.948099,7.731282,2.032315,47230.996869,14.120724
4,5,932,0.278641,192.784335,10.148482,9458.385403,2.827788


Ratio mayoria/minoria: 179,675 / 932 = 192.8:1
Acuity 1-2 en train: 39.06%
Acuity 4-5 en train: 7.23%
Acuity 5 por fold aprox. en CV de 5 folds: 186.4 casos


## 4. Helpers de metricas

Optuna optimiza Macro F1. Tambien se guardan precision y recall de Acuity 1 y 2 para asegurar que una mejora global no empeora de forma preocupante las clases mas sensibles.

In [4]:
def metricas_oof(y_true: pd.Series | np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=CLASSES, zero_division=0
    )
    return {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_a1": float(recall[0]),
        "precision_a1": float(precision[0]),
        "recall_a2": float(recall[1]),
        "precision_a2": float(precision[1]),
        "f1_a1": float(f1[0]),
        "f1_a2": float(f1[1]),
    }


def fixed_lgbm_params() -> dict[str, Any]:
    return {
        "objective": "multiclass",
        "num_class": 5,
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbose": -1,
        "subsample_freq": 1,
    }


def suggest_lgbm_params(trial: optuna.Trial) -> dict[str, Any]:
    params = fixed_lgbm_params()
    params.update(
        {
        "num_leaves": trial.suggest_int("num_leaves", 48, 192),
        "max_depth": trial.suggest_int("max_depth", 5, 12),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 120),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 3.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 8.0, log=True),
        "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 2.0),
        "subsample": trial.suggest_float("subsample", 0.70, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 0.95),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.06, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 350, 1200),
        }
    )
    return params


def params_from_completed_trial(trial: optuna.trial.FrozenTrial) -> dict[str, Any]:
    params = fixed_lgbm_params()
    params.update(trial.params)
    return params


def evaluate_params(params: dict[str, Any], trial: optuna.Trial | None = None) -> dict[str, Any]:
    cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=False)
    oof_pred = np.zeros(len(y_train), dtype=int)
    fold_scores: list[float] = []
    best_iterations: list[int] = []

    for fold, (idx_tr, idx_val) in enumerate(cv.split(X_train, y_train, groups_train), start=1):
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train.iloc[idx_tr],
            y_train.iloc[idx_tr] - 1,
            sample_weight=sample_weights_train[idx_tr],
            eval_set=[(X_train.iloc[idx_val], y_train.iloc[idx_val] - 1)],
            eval_sample_weight=[sample_weights_train[idx_val]],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        pred = model.predict(X_train.iloc[idx_val]) + 1
        oof_pred[idx_val] = pred
        fold_macro = float(f1_score(y_train.iloc[idx_val], pred, average="macro", zero_division=0))
        fold_scores.append(fold_macro)
        best_iterations.append(int(model.best_iteration_ or params["n_estimators"]))

        if trial is not None:
            trial.report(float(np.mean(fold_scores)), step=fold)
            if trial.should_prune():
                raise optuna.TrialPruned()

    metrics = metricas_oof(y_train, oof_pred)
    metrics.update(
        {
            "fold_macro_f1_mean": float(np.mean(fold_scores)),
            "fold_macro_f1_std": float(np.std(fold_scores)),
            "best_iteration_mean": float(np.mean(best_iterations)),
            "best_iteration_std": float(np.std(best_iterations)),
        }
    )
    return metrics

## 5. Busqueda Optuna

El estudio se guarda en SQLite para poder reanudarlo si se interrumpe. Cada trial entrena cinco folds y calcula OOF sobre train.

In [5]:
def objective(trial: optuna.Trial) -> float:
    params = suggest_lgbm_params(trial)
    start = time.perf_counter()
    metrics = evaluate_params(params, trial=trial)
    elapsed = time.perf_counter() - start

    for key, value in metrics.items():
        trial.set_user_attr(key, value)
    trial.set_user_attr("elapsed_seconds", float(elapsed))

    return metrics["macro_f1"]


sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True)
pruner = optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=2)
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_URL,
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    load_if_exists=True,
)

study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SECONDS, show_progress_bar=True)

print("Best trial:", study.best_trial.number)
print("Best Macro F1:", study.best_value)
print("Best params:")
print(json.dumps(study.best_trial.params, indent=2))

[I 2026-06-06 18:12:03,928] Using an existing study with name 'lgbm_bert_optuna_macro_f1_oof' instead of creating a new one.
Best trial: 1. Best value: 0.566552:   2%|▏         | 1/60 [04:29<4:24:41, 269.19s/it]

[I 2026-06-06 18:16:33,102] Trial 1 finished with value: 0.5665522057115873 and parameters: {'num_leaves': 102, 'max_depth': 12, 'min_data_in_leaf': 91, 'reg_alpha': 0.04789240251631178, 'reg_lambda': 0.004064010237521863, 'min_gain_to_split': 0.3119890406724053, 'subsample': 0.7174250836504598, 'colsample_bytree': 0.8897792655987208, 'learning_rate': 0.0293601586511158, 'n_estimators': 952}. Best is trial 1 with value: 0.5665522057115873.


Best trial: 2. Best value: 0.567349:   3%|▎         | 2/60 [10:05<4:58:08, 308.42s/it]

[I 2026-06-06 18:22:08,988] Trial 2 finished with value: 0.5673487483470393 and parameters: {'num_leaves': 50, 'max_depth': 12, 'min_data_in_leaf': 102, 'reg_alpha': 0.0008926227381843736, 'reg_lambda': 0.005124826994072942, 'min_gain_to_split': 0.36680901970686763, 'subsample': 0.7912726728878613, 'colsample_bytree': 0.7361403942345071, 'learning_rate': 0.021682959386691958, 'n_estimators': 597}. Best is trial 2 with value: 0.5673487483470393.


Best trial: 2. Best value: 0.567349:   5%|▌         | 3/60 [13:17<4:02:41, 255.46s/it]

...

[I 2026-06-06 23:22:42,637] Trial 58 pruned. 


Best trial: 50. Best value: 0.570053:  98%|█████████▊| 59/60 [5:14:33<03:50, 230.10s/it]

[I 2026-06-06 23:26:37,381] Trial 59 pruned. 


Best trial: 50. Best value: 0.570053: 100%|██████████| 60/60 [5:17:34<00:00, 317.57s/it]

[I 2026-06-06 23:29:38,408] Trial 60 pruned. 
Best trial: 50
Best Macro F1: 0.5700533452742491
Best params:
{
  "num_leaves": 118,
  "max_depth": 7,
  "min_data_in_leaf": 40,
  "reg_alpha": 0.1885210600468343,
  "reg_lambda": 0.18245157688621794,
  "min_gain_to_split": 0.3205376221177295,
  "subsample": 0.8313831030674474,
  "colsample_bytree": 0.5783589264118796,
  "learning_rate": 0.012102188789764785,
  "n_estimators": 1036
}


## 6. Exportacion de trials y parametros

Los resultados se guardan separados del modelo final congelado. Este notebook no sobrescribe `models/lgbm_bert_final.joblib`.

In [6]:
trials_df = study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs", "duration"))
trials_path = OUT_DIR / "lgbm_bert_optuna_trials.csv"
trials_df.to_csv(trials_path, index=False, encoding="utf-8")

best_params = params_from_completed_trial(study.best_trial)
best_params["n_estimators"] = int(round(study.best_trial.user_attrs.get("best_iteration_mean", best_params["n_estimators"])))

best_payload = {
    "study_name": STUDY_NAME,
    "baseline_macro_f1_oof": BASELINE_MACRO_F1,
    "best_trial_number": int(study.best_trial.number),
    "best_macro_f1_oof": float(study.best_value),
    "improvement_vs_baseline": float(study.best_value - BASELINE_MACRO_F1),
    "best_params_raw_trial": study.best_trial.params,
    "best_params_lgbm": best_params,
    "best_user_attrs": study.best_trial.user_attrs,
    "n_trials_total_in_study": len(study.trials),
    "test_usage": "No se carga ni se usa test temporal en este notebook.",
}

best_path = OUT_DIR / "lgbm_bert_optuna_best_params.json"
best_path.write_text(json.dumps(best_payload, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Trials -> {trials_path}")
print(f"Best params -> {best_path}")

Trials -> C:\Users\CARLOS\triaje-ia-tfg\reports\hyperparameter_tuning\lgbm_bert_optuna_trials.csv
Best params -> C:\Users\CARLOS\triaje-ia-tfg\reports\hyperparameter_tuning\lgbm_bert_optuna_best_params.json


## 7. Comparacion contra el modelo congelado

La comparacion se hace contra el resultado OOF ya congelado en el notebook 07. Si la mejora no es clara o empeora clases sensibles, se mantiene el modelo actual.

In [ ]:
baseline_row = {
    "modelo": "lightgbm_final_bert_actual",
    "macro_f1": 0.567940,
    "recall_a1": 0.687523,
    "precision_a1": 0.652363,
    "recall_a2": 0.654815,
    "precision_a2": 0.669934,
    "best_iteration_mean": 491.2,
}

best_attrs = study.best_trial.user_attrs
optuna_row = {
    "modelo": "lightgbm_final_bert_optuna",
    "macro_f1": float(study.best_value),
    "recall_a1": best_attrs.get("recall_a1"),
    "precision_a1": best_attrs.get("precision_a1"),
    "recall_a2": best_attrs.get("recall_a2"),
    "precision_a2": best_attrs.get("precision_a2"),
    "best_iteration_mean": best_attrs.get("best_iteration_mean"),
}

comparison = pd.DataFrame([baseline_row, optuna_row])
comparison["delta_macro_f1_vs_actual"] = comparison["macro_f1"] - BASELINE_MACRO_F1
comparison

,modelo,macro_f1,recall_a1,precision_a1,recall_a2,precision_a2,best_iteration_mean,delta_macro_f1_vs_actual
0,lightgbm_final_bert_actual,0.567940,0.687523,0.652363,0.654815,0.669934,491.2,0.000000
1,lightgbm_final_bert_optuna,0.570053,0.685767,0.650698,0.653305,0.668943,1035.6,0.002113


: 

In [ ]:
summary_path = OUT_DIR / "lgbm_bert_optuna_summary.md"
improvement = float(study.best_value - BASELINE_MACRO_F1)
decision = (
    "La configuracion Optuna mejora el OOF del modelo actual y puede considerarse candidata, "
    "pendiente de congelacion y evaluacion final controlada."
    if improvement > 0.001
    else "La mejora no es suficientemente clara; se mantiene el modelo congelado actual."
)

comparison_md = comparison.to_csv(index=False, float_format="%.6f")

lines = [
    "# Busqueda Optuna LightGBM+BERT/SVD",
    "",
    "Busqueda de hiperparametros realizada exclusivamente sobre train con validacion cruzada agrupada por paciente. El test temporal no se ha usado para seleccionar parametros.",
    "",
    "## Configuracion",
    "",
    f"- Estudio: `{STUDY_NAME}`",
    f"- Trials totales en el estudio: {len(study.trials)}",
    f"- Folds: {N_SPLITS}",
    "- Metrica objetivo: Macro F1 OOF",
    "- Modelo: LightGBM + variables tabulares finales + BERT/SVD",
    "",
    "## Comparacion",
    "",
    "```csv",
    comparison_md.strip(),
    "```",
    "",
    "## Mejor trial",
    "",
    f"- Trial: {study.best_trial.number}",
    f"- Macro F1 OOF: {study.best_value:.6f}",
    f"- Mejora frente al actual: {improvement:+.6f}",
    f"- Recall A1: {best_attrs.get('recall_a1', float('nan')):.6f}",
    f"- Precision A1: {best_attrs.get('precision_a1', float('nan')):.6f}",
    f"- Recall A2: {best_attrs.get('recall_a2', float('nan')):.6f}",
    f"- Precision A2: {best_attrs.get('precision_a2', float('nan')):.6f}",
    "",
    "## Lectura metodologica",
    "",
    decision,
    "",
    "No se sobrescribe el modelo final actual. Si esta configuracion se adopta, debe congelarse primero con todo train y solo despues evaluarse una vez en test temporal.",
    "",
    "## Parametros ganadores",
    "",
    "```json",
    json.dumps(best_params, ensure_ascii=False, indent=2),
    "```",
    "",
    "## Artefactos",
    "",
    f"- Trials: `{trials_path.relative_to(PROJECT_ROOT)}`",
    f"- Parametros: `{best_path.relative_to(PROJECT_ROOT)}`",
    f"- Resumen: `{summary_path.relative_to(PROJECT_ROOT)}`",
    "",
    "## Nota para la memoria",
    "",
    "Este experimento puede citarse como una prueba adicional de ajuste posterior a la seleccion de arquitectura. La seleccion sigue basandose en OOF/train, no en test.",
]
summary_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Resumen -> {summary_path}")

## 9. Cierre

Este notebook deja documentada la busqueda de hiperparametros del ganador. El siguiente paso solo debe ejecutarse si se decide adoptar el candidato: reentrenar con todo train, congelar artefactos nuevos y evaluar una unica vez en el test temporal.